# 📈 Sales Prediction Using Python
### CodeAlpha Data Science Internship – Task 3

**Project goal:** Predict sales using advertising expenditure, advertising platform, and target customer segment.

**Dataset note:** This project uses a reproducible, illustrative synthetic dataset created for learning and demonstration. It is not collected from a real company. The dataset includes 500 campaign records.

**Workflow:** Load data → inspect and clean → explore and visualize → encode categorical features → train regression models → evaluate → predict sales → derive business insights.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")


## 2. Load the dataset

In [ ]:
# Upload CodeAlpha_Task4_Sales_Dataset.csv in Google Colab when prompted
from google.colab import files
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())


## 3. Understand and inspect the data

In [ ]:
print("Dataset information:")
df.info()

print("\\nMissing values:")
print(df.isnull().sum())

print("\\nDuplicate rows:", df.duplicated().sum())
display(df.describe(include="all"))


## 4. Data cleaning

In [ ]:
# Remove duplicate rows and rows without a target value
df = df.drop_duplicates()
df = df.dropna(subset=["Sales"]).copy()

# Fill missing numerical values with the median and categorical values with the mode
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Cleaning completed.")
print("Remaining missing values:", int(df.isnull().sum().sum()))


## 5. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df["Sales"], kde=True)
plt.title("Distribution of Sales")
plt.xlabel("Sales")
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="Platform", y="Sales")
plt.title("Sales by Advertising Platform")
plt.xticks(rotation=15)
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="TV_Ad_Spend", y="Sales", hue="Platform")
plt.title("TV Advertising Spend vs Sales")
plt.show()


## 6. Compare average sales by platform and target segment

In [ ]:
platform_sales = df.groupby("Platform")["Sales"].mean().sort_values(ascending=False)
segment_sales = df.groupby("Target_Segment")["Sales"].mean().sort_values(ascending=False)

print("Average sales by platform:")
display(platform_sales.to_frame("Average_Sales"))

print("Average sales by target segment:")
display(segment_sales.to_frame("Average_Sales"))

platform_sales.plot(kind="bar", figsize=(7, 4), title="Average Sales by Platform", ylabel="Average Sales")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 7. Prepare features and target

In [ ]:
# Features: advertising budgets, platform, and customer segment
X = df[[
    "TV_Ad_Spend",
    "Social_Media_Ad_Spend",
    "Search_Ad_Spend",
    "Platform",
    "Target_Segment"
]]
y = df["Sales"]

numeric_features = ["TV_Ad_Spend", "Social_Media_Ad_Spend", "Search_Ad_Spend"]
categorical_features = ["Platform", "Target_Segment"]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])


## 8. Train the regression model

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=200, random_state=42))
])

model.fit(X_train, y_train)
print("Random Forest Regression model trained successfully!")


## 9. Evaluate model performance

In [ ]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R²): {r2:.4f}")


## 10. Actual vs predicted sales

In [ ]:
results = pd.DataFrame({
    "Actual_Sales": y_test.values,
    "Predicted_Sales": predictions
})
display(results.head(10))

plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=predictions)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales")
low = min(y_test.min(), predictions.min())
high = max(y_test.max(), predictions.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.show()


## 11. Predict sales for a new campaign

In [ ]:
new_campaign = pd.DataFrame([{
    "TV_Ad_Spend": 6000,
    "Social_Media_Ad_Spend": 4500,
    "Search_Ad_Spend": 3000,
    "Platform": "Instagram",
    "Target_Segment": "Young Adults"
}])

predicted_sales = model.predict(new_campaign)[0]
print(f"Predicted sales for the new campaign: {predicted_sales:.2f}")


## 12. Feature importance

In [ ]:
# Extract feature names after one-hot encoding
encoded_names = model.named_steps["preprocessor"].get_feature_names_out()
importance = model.named_steps["regressor"].feature_importances_

feature_importance = pd.Series(importance, index=encoded_names).sort_values(ascending=False)
display(feature_importance.to_frame("Importance").head(10))

feature_importance.head(10).sort_values().plot(kind="barh", figsize=(8, 5), title="Top 10 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 13. Business insights and conclusion

Use the actual charts and metrics produced above to discuss:
- Which advertising platform has the highest average sales in this dataset.
- Which target segment has the highest average sales.
- Which advertising budget or category is most influential according to the model.
- How close predicted sales are to actual sales, using MAE and R².

**Conclusion:** This project demonstrates a complete sales-prediction workflow using Python and Random Forest Regression. The model estimates sales from advertising budgets, platform, and target segment. Since the dataset is synthetic and illustrative, the results demonstrate the workflow and should not be treated as real business forecasts.


## 14. Save the cleaned dataset and model (optional)

In [ ]:
df.to_csv("cleaned_sales_dataset.csv", index=False)
print("Cleaned dataset saved successfully.")

import joblib
joblib.dump(model, "sales_prediction_model.pkl")
print("Trained model saved successfully.")
